# AeroPulse Source Likelihood — 05: Source Classifier Training

One binary classifier per source, four model families compared under one
contract, on a feature set that cannot see the labeling functions.

## Objective

Improvement plan sections 21, 24 and 27.

Section 24 is a hard architectural boundary and this notebook is where it
is enforced: the classifier must not see the columns the labeling functions
read, or it learns the heuristic rather than the signal. `select_features`
excludes every labeling-function input *and every column those inputs were
derived from* — `no2_level_anomaly` is `cams_no2` divided by a per-station
constant, so leaving `cams_no2` in would hand the model the same
information under another name.

Section 21 asks for a comparison across Logistic Regression, LightGBM,
XGBoost and CatBoost per source. Section 27 governs how to read the result:

> the model reproduces the weak-labeling system

A high PR-AUC here means the classifier learned to predict what the
heuristics say from evidence the heuristics could not see. That is a real
and useful thing to measure — it tells us whether the source signal is
present in independent evidence — but it is **not** attribution accuracy,
and notebook 08 is where that distinction gets its own measurement.


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
df = pd.read_parquet(cfg.feature_dir / "source_evidence_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
labels = pd.read_parquet(cfg.label_dir / "weak_labels.parquet")
work = df.join(labels)

FEATURES = st.select_features(work)
st.assert_no_label_leakage(FEATURES)
print("rows", f"{len(work):,}", "| leak-safe features", len(FEATURES))

def subsample(frame, cap):
    if cap is None or len(frame) <= cap:
        return frame
    return frame.iloc[(np.arange(cap) * (len(frame) / cap)).astype(int)]

CAP = cfg.max_train_rows or 400_000
train, valid, test = st.chrono_split(work, cfg.train_fraction, cfg.validation_fraction)
train = subsample(train, CAP)
print(f"train {len(train):,}  valid {len(valid):,}  test {len(test):,}")

rows 1,705,252 | leak-safe features 56


train 400,000  valid 255,833  test 255,807


## 1. What the classifier is allowed to see

In [3]:
print(f"labeling-function inputs excluded ({len(st.LF_INPUT_COLUMNS)}):")
for column in st.LF_INPUT_COLUMNS:
    print("   ", column)
print(f"\nfeatures available to the classifier ({len(FEATURES)}):")
for i in range(0, len(FEATURES), 4):
    print("   ", "  ".join(f"{c:<28}" for c in FEATURES[i:i + 4]))

labeling-function inputs excluded (28):
    cams_dust
    cams_no2
    cams_no2_station_median
    cams_pm10
    cams_pm10_station_median
    cams_so2
    cams_so2_station_median
    coarse_fraction_anomaly
    crop_burning_window
    fire_active_flag
    fire_count_20km_24h
    hour
    is_weekend
    no2_level_anomaly
    no2_pm25_ratio
    pm10_pm25_ratio
    pm25
    relative_humidity_2m
    rolling_corr_no2_pm25
    season
    so2_level_anomaly
    so2_pm25_ratio
    stagnation_flag
    station_weekend_ratio
    transport_direction_consistency
    upwind_fire_frp_50km
    upwind_pm25_mean
    wind_speed_10m

features available to the classifier (56):
    temperature_2m                surface_pressure              precipitation                 boundary_layer_height       
    rain_1h                       rain_3h                       rain_6h                       rain_12h                    
    rain_24h                      rain_48h                      delta_temperature_3h      

In [4]:
# Negative control: the leakage guard must fire when it should.
try:
    st.assert_no_label_leakage(FEATURES + ["pm25"])
    print("FAIL — leakage guard did not fire")
except AssertionError as exc:
    print("PASS — guard rejected a feature set containing pm25:")
    print("  ", str(exc)[:160])

PASS — guard rejected a feature set containing pm25:
   label inputs leaked: ['pm25']; label outputs leaked: []; same-family evidence leaked: ['pm25']


## 2. Label balance per source in each split

In [5]:
rows = []
for source in SOURCES:
    entry = {"source": source}
    for name, part in (("train", train), ("valid", valid), ("test", test)):
        y = part[f"y_{source}"]
        entry[f"{name}_pos"] = int(y.sum())
        entry[f"{name}_rate"] = round(float(y.mean()) * 100, 2)
    entry["trainable"] = bool(min(int(train[f"y_{source}"].sum()),
                                  int((1 - train[f"y_{source}"]).sum())) > 100)
    rows.append(entry)
balance = pd.DataFrame(rows)
print(balance.to_string(index=False))

TRAINABLE = balance.loc[balance["trainable"], "source"].tolist()
print("\ntrainable sources:", TRAINABLE)
skipped = [s for s in SOURCES if s not in TRAINABLE]
if skipped:
    print("skipped (single-class or too few positives in train):", skipped)

            source  train_pos  train_rate  valid_pos  valid_rate  test_pos  test_rate  trainable
   biomass_burning      32048        8.01      13406        5.24       172       0.07       True
           traffic     173994       43.50      85153       33.28     54862      21.45       True
        industrial      65666       16.42      19043        7.44     10534       4.12       True
              dust     118601       29.65     123296       48.19     92792      36.27       True
regional_transport      21888        5.47       4090        1.60      1177       0.46       True

trainable sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']


## 3. Model family comparison (section 21)

Same features, same split, same target per source. PR-AUC is the headline
because the positive rates differ by an order of magnitude between sources;
`pr_auc_lift` divides it by prevalence so the numbers are comparable.

In [6]:
FAMILIES = ["logistic", "lightgbm", "xgboost", "catboost"]
comparison_rows = []
for source in TRAINABLE:
    for kind in FAMILIES:
        t0 = time.time()
        try:
            model = st.SourceClassifier.fit(train, FEATURES, source, kind=kind)
            proba = model.raw_proba(valid)
            metrics = st.calibration_metrics(valid[f"y_{source}"].to_numpy(), proba)
        except Exception as exc:
            print(f"  {source:<20} {kind:<10} FAILED: {exc}")
            continue
        comparison_rows.append({
            "source": source, "family": kind,
            "pr_auc": round(metrics["pr_auc"], 4),
            "pr_auc_lift": round(metrics["pr_auc_lift"], 3),
            "roc_auc": round(metrics["roc_auc"], 4),
            "brier": round(metrics["brier"], 4),
            "ece": round(metrics["ece"], 4),
            "fit_seconds": round(time.time() - t0, 1),
        })
        print(f"  {source:<20} {kind:<10} PR-AUC {metrics['pr_auc']:.4f} "
              f"(lift {metrics['pr_auc_lift']:.2f}x)  ECE {metrics['ece']:.4f}  "
              f"{time.time() - t0:.0f}s")

families = pd.DataFrame(comparison_rows)
print()
print(families.pivot_table(index="source", columns="family", values="pr_auc_lift").round(3).to_string())

  biomass_burning      logistic   PR-AUC 0.1268 (lift 2.42x)  ECE 0.1394  3s


  biomass_burning      lightgbm   PR-AUC 0.2805 (lift 5.35x)  ECE 0.1249  6s


  biomass_burning      xgboost    PR-AUC 0.2734 (lift 5.22x)  ECE 0.0119  3s


  biomass_burning      catboost   PR-AUC 0.2636 (lift 5.03x)  ECE 0.0071  7s


  traffic              logistic   PR-AUC 0.6983 (lift 2.10x)  ECE 0.0751  1s


...

  dust                 catboost   PR-AUC 0.8421 (lift 1.75x)  ECE 0.0241  7s


  regional_transport   logistic   PR-AUC 0.0279 (lift 1.75x)  ECE 0.1713  2s


  regional_transport   lightgbm   PR-AUC 0.0655 (lift 4.10x)  ECE 0.1085  8s


  regional_transport   xgboost    PR-AUC 0.0644 (lift 4.03x)  ECE 0.0022  3s


  regional_transport   catboost   PR-AUC 0.0782 (lift 4.89x)  ECE 0.0023  7s

family              catboost  lightgbm  logistic  xgboost
source                                                   
biomass_burning        5.030     5.354     2.421    5.218
dust                   1.747     1.754     1.639    1.754
industrial             4.583     5.016     3.044    4.997
regional_transport     4.893     4.098     1.745    4.027
traffic                2.246     2.280     2.098    2.279


In [7]:
ranking = (families.groupby("family")
           .agg(mean_lift=("pr_auc_lift", "mean"),
                mean_pr_auc=("pr_auc", "mean"),
                mean_ece=("ece", "mean"),
                mean_fit_s=("fit_seconds", "mean"))
           .sort_values("mean_lift", ascending=False).round(4))
print(ranking.to_string())
WINNER = ranking.index[0]
print("\nselected family:", WINNER)

          mean_lift  mean_pr_auc  mean_ece  mean_fit_s
family                                                
lightgbm     3.7004       0.4647    0.0934        7.42
catboost     3.6998       0.4545    0.0118        7.08
xgboost      3.6550       0.4627    0.0144        2.96
logistic     2.1894       0.3739    0.1359        1.76

selected family: lightgbm


## 4. Is the model learning signal, or memorising the label prior?

A classifier can score well simply by predicting the base rate in each
regime. Compare against two null models: a constant predictor at the
training prevalence, and a shuffled-label control that destroys any real
relationship while keeping the marginal distribution.

In [8]:
null_rows = []
for source in TRAINABLE:
    y_valid = valid[f"y_{source}"].to_numpy()
    prevalence = float(train[f"y_{source}"].mean())

    constant = st.calibration_metrics(y_valid, np.full(len(y_valid), prevalence))
    shuffled_train = train.copy()
    rng = np.random.default_rng(0)
    shuffled_train[f"y_{source}"] = rng.permutation(shuffled_train[f"y_{source}"].to_numpy())
    shuffled_model = st.SourceClassifier.fit(shuffled_train, FEATURES, source, kind=WINNER)
    shuffled = st.calibration_metrics(y_valid, shuffled_model.raw_proba(valid))

    real = families[(families["source"] == source) & (families["family"] == WINNER)].iloc[0]
    null_rows.append({
        "source": source,
        "model_pr_auc": real["pr_auc"],
        "constant_pr_auc": round(constant["pr_auc"], 4),
        "shuffled_pr_auc": round(shuffled["pr_auc"], 4),
        "lift_over_shuffled": round(float(real["pr_auc"] / max(shuffled["pr_auc"], 1e-6)), 2),
    })
nulls = pd.DataFrame(null_rows)
print(nulls.to_string(index=False))

            source  model_pr_auc  constant_pr_auc  shuffled_pr_auc  lift_over_shuffled
   biomass_burning        0.2805           0.0524           0.0546                5.14
           traffic        0.7589           0.3328           0.3182                2.38
        industrial        0.3734           0.0744           0.0704                5.30
              dust        0.8451           0.4819           0.4870                1.74
regional_transport        0.0655           0.0160           0.0161                4.07


## 5. Which evidence drives each source?

In [9]:
models = {}
importances = {}
for source in TRAINABLE:
    model = st.SourceClassifier.fit(train, FEATURES, source, kind=WINNER)
    models[source] = model
    estimator = model.model
    inner = estimator[-1] if hasattr(estimator, "__getitem__") and hasattr(estimator, "steps") else estimator
    if hasattr(inner, "feature_importances_"):
        values = np.asarray(inner.feature_importances_, dtype=float)
    elif hasattr(inner, "coef_"):
        values = np.abs(np.asarray(inner.coef_, dtype=float)).ravel()
    else:
        values = np.zeros(len(FEATURES))
    imp = pd.DataFrame({"feature": FEATURES, "importance": values})
    imp["importance_pct"] = imp["importance"] / max(imp["importance"].sum(), 1e-9) * 100
    importances[source] = imp.sort_values("importance", ascending=False).reset_index(drop=True)
    print(f"\n{source}: top evidence")
    for _, r in importances[source].head(6).iterrows():
        print(f"    {r['feature']:<34} {r['importance_pct']:5.2f}%")


biomass_burning: top evidence
    lon                                 4.56%
    lat                                 4.46%
    station_night_ratio                 4.40%
    sin_doy                             4.36%
    doy                                 4.19%
    cos_doy                             3.76%



traffic: top evidence
    lat                                 4.90%
    lon                                 4.50%
    cams_o3                             4.19%
    station_morning_ratio               4.13%
    cos_doy                             4.04%
    cams_co_station_median              3.72%



industrial: top evidence
    lat                                 5.91%
    lon                                 5.67%
    cams_o3                             4.34%
    sin_doy                             3.94%
    cos_doy                             3.84%
    cams_co_station_median              3.75%



dust: top evidence
    lat                                 6.04%
    sin_doy                             5.94%
    doy                                 5.89%
    lon                                 5.66%
    cos_doy                             4.37%
    cams_co_station_median              3.92%



regional_transport: top evidence
    ventilation_coefficient             5.05%
    lat                                 5.04%
    lon                                 4.23%
    delta_pbl_3h                        3.30%
    delta_temperature_3h                3.21%
    sin_doy                             3.11%


## 6. Save

In [10]:
import joblib
stage = cfg.stage_dir("classifiers")
families.to_csv(stage / "family_comparison.csv", index=False)
balance.to_csv(stage / "label_balance.csv", index=False)
nulls.to_csv(stage / "null_model_comparison.csv", index=False)
for source, imp in importances.items():
    imp.to_csv(stage / f"feature_importance_{source}.csv", index=False)
joblib.dump({"models": models, "features": FEATURES, "winner": WINNER},
            stage / "source_classifiers.joblib")

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "winner_family": WINNER,
    "features": FEATURES,
    "n_features": len(FEATURES),
    "lf_inputs_excluded": list(st.LF_INPUT_COLUMNS),
    "trainable_sources": TRAINABLE,
    "skipped_sources": skipped,
    "family_comparison": families.to_dict(orient="records"),
    "family_ranking": ranking.reset_index().to_dict(orient="records"),
    "null_models": nulls.to_dict(orient="records"),
    "caveat": "measures reproduction of the weak-labeling system, not attribution accuracy",
}
(stage / "classifier_report.json").write_text(json.dumps(report, indent=2, default=str))
print("05 COMPLETE ->", stage)
print("Next: 06_source_probability_calibration.ipynb")

05 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/artifacts/source/classifiers
Next: 06_source_probability_calibration.ipynb


## Findings

**The first run of this notebook produced PR-AUC 0.9999 for biomass burning
and a 62x PR-AUC lift for regional transport. Those numbers were a leakage
report, not a result.** They are recorded here because the failure mode is
the one the plan's section 27 is about, and because the numbers looked like
success.

The exclusion list named the exact labeling-function inputs but not their
evidence families. The classifier was therefore reading:

- `n_sources_evaluated` — a label-aggregation *output*, which ranked in the
  top five for every single source
- `upwind_fire_count_50km` — a near-duplicate of the fire LF's
  `upwind_fire_frp_50km`
- `upwind_pm25_gradient`, `upwind_pm25_100km`, `upwind_pm25_max` — near
  duplicates of the transport LF's `upwind_pm25_mean`
- `pm25_lag_1h`, `pm25_local_excess` — near-duplicates of `pm25`, which
  every high-PM gate reads

Exclusion is now family-level: 42 prefixes covering every evidence family
any labeling function touches, plus all `*_pm25_ratio` columns and the
`evidence_*` availability flags (those determine whether an LF abstains, so
they predict the label through the labeling mechanism itself). The candidate
set fell from 247 columns to **56 genuinely independent features**: weather,
rain, CAMS CO/O3/AOD, calendar, terrain, and the station behavioural proxies.

**With the leak closed, the honest numbers are 1.7x to 5.4x PR-AUC lift.**

| source | PR-AUC | lift | vs shuffled control |
|---|---|---|---|
| dust | 0.845 | 1.74x | 1.74x |
| traffic | 0.759 | 2.28x | 2.38x |
| industrial | 0.373 | 5.02x | 5.30x |
| biomass_burning | 0.281 | 5.35x | 5.14x |
| regional_transport | 0.066 | 4.07x | 4.07x |

Every source beats both null models — a constant predictor at the training
prevalence and a shuffled-label control — so there is real signal in
evidence the labeling functions never saw. That is the useful finding: the
source hypotheses are not purely definitional.

**But look at what the models key on.** For four of five sources the top
features are `lat`, `lon` and day-of-year:

| source | top three |
|---|---|
| biomass_burning | lon, lat, station_night_ratio |
| traffic | lat, lon, cams_o3 |
| industrial | lat, lon, cams_o3 |
| dust | lat, sin_doy, doy |
| regional_transport | ventilation_coefficient, lat, lon |

The classifier has largely learned **where and when**, not source physics.
That is a coherent thing to learn — stubble burning really is a Punjab
autumn phenomenon and dust really is a pre-monsoon western one — and it
would generalise poorly to a new region or an unusual season. Notebook 07
tests exactly that. Only `regional_transport` leads with a physical
variable (`ventilation_coefficient`), which is also the source with the
weakest absolute performance.

**Gradient boosting beats logistic regression decisively**, unlike the MVP's
assumption: mean lift 19.6 for LightGBM against 12.8 for logistic in the
leaked run, and in the clean run logistic trails on every source
(biomass 2.42x vs 5.35x, industrial 3.04x vs 5.02x). LightGBM, XGBoost and
CatBoost are within 3% of each other; LightGBM was selected on mean lift.
Worth noting XGBoost produced far better-calibrated raw probabilities
(ECE 0.0119 vs 0.1249 on biomass), which notebook 06 addresses separately.

**Label prevalence is wildly unstable across the split**, and this governs
how every later number must be read: biomass burning is 8.01% of training
rows and **0.07% of test rows**; traffic falls 43.50% to 21.45%; dust rises
29.65% to 36.27%. The weak labels are seasonal and the split is
chronological, so the test period is asking a materially different question.
PR-AUC lift, not PR-AUC, is the only comparable quantity.